> **Documentation / analysis only.** The production path for this step is the headless pipeline:
> `python -m ueba.pipeline preprocess`
>
> This notebook is kept as the narrative record of the training procedure and for interactive
> exploration. The pipeline stages call the same `ueba.*` package functions, validate their
> input artifacts through a fail-fast manifest, and persist evaluation metrics as JSON.
> See `docs/PIPELINE.md`.

# **Preprocessing the CERT Dataset**:

### Imports & Constants:

In [1]:
import config
from scripts.Preprocessing import *
from prepare_data import get_insiders, build_insider_mask
from config import CERT_PATH, DEFAULT_OUTPUT_DIR
from config import INSIDERS_PATH, UEBA_CALIBRATION_PATH, UEBA_CALIB_EVAL_PATH

In [3]:
MV = config.MODEL_VERSION
print("Model Version:", MV)
WORK_HOURS = (9, 17)
LONG_URL_THRESHOLD = 90

Model Version: 6


### Building Layer A: (User, PC, Day) Level Dataset

In [ ]:
layer_a_dataset, nunique_frames = build_layer_a(
    cert_path=CERT_PATH,
    work_hours=WORK_HOURS,
    return_nunique_frames=True,
    compute_schedules=True,
    save_schedule_to=os.path.join(DEFAULT_OUTPUT_DIR, "user_work_hours.parquet")
)

In [ ]:
layer_a_dataset.head()

In [ ]:
layer_a_path = save_dataset(
    dataset=layer_a_dataset,
    filename=f"ueba_dataset_{MV}a.parquet",
    output_dir=DEFAULT_OUTPUT_DIR
)

In [ ]:
# Saving nunique frames for recovery
save_nunique_frames(nunique_frames, os.path.join(DEFAULT_OUTPUT_DIR, "safepoint"))

### Resuming from Safepoint:

In [4]:
# Reload layer_a_dataset and nunique_frames from disk — skip build_layer_a
layer_a_dataset = pd.read_parquet(os.path.join(DEFAULT_OUTPUT_DIR, f"ueba_dataset_{MV}a.parquet"))
nunique_frames = load_nunique_frames(os.path.join(DEFAULT_OUTPUT_DIR, "safepoint"))

Safepoint loaded: 3 nunique frames from processed_datasets\ueba_dataset_6\safepoint


### Building Layer B: (User, Day) Model Training Dataset

In [5]:
# Loading the LDAP metadata
ldap_df = load_ldap(CERT_PATH)

Loaded 18 LDAP snapshots: 4,000 unique users, 46 unique roles, 68,923 total rows.


In [6]:
layer_b_dataset = build_layer_b(
    layer_a_df=layer_a_dataset,
    rolling_window=5,
    nunique_frames=nunique_frames,
    ldap_df=ldap_df,
    peer_col=config.PEER_GROUP_KEY,
)

Collapsing Layer A to (user, day) granularity...
  Aggregating per-PC behavioral features...
  Computing cross-channel flags...
  Recomputing true nunique counts from raw event frames...
Adding multi-horizon rolling features (7d/30d sums, 1d-over-30d ratios)...
Applying UEBA enhancements (z-scores, rolling deltas, risk flags)...
  Computing per-user causal z-score deviations...
  Computing per-user long-horizon (90d) z-score deviations...


c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1726: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[z_scores_90.columns] = z_scores_90
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1726: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[z_scores_90.columns] = z_scores_90
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1726: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

  Computing causal rolling mean deltas...


c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1737: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_deltas.columns] = rolling_deltas
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1737: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_deltas.columns] = rolling_deltas
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1737: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

  Adding cross-channel risk flags...
Applying peer-group enhancements (peer_col='role')...


c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1875: RuntimeWarning: divide by zero encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1875: RuntimeWarning: invalid value encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1875: RuntimeWarning: divide by zero encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1875: RuntimeWarning: invalid value encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1875: RuntimeWarning: divide by zero encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1875: RuntimeWarning: invalid value encountered in d

Joining LDAP user profiles (employee_name, department, role, supervisor, functional_unit, role_sensitivity, is_active)...
Layer B complete — 1,394,010 rows, 414 features.


In [7]:
layer_b_dataset.head()

,user,day,logon_count,logoff_count,off_hours_logon,logon_late_night_count,file_open_count,file_write_count,file_copy_count,file_delete_count,...,non_primary_pc_http_requests_flag_peer_zscore,non_primary_pc_usb_flag_peer_zscore,non_primary_pc_file_copy_flag_peer_zscore,employee_name,role,department,functional_unit,supervisor,is_active,role_sensitivity
0,aab0162,2010-01-04,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,Amos Ahmed Burch,HardwareEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,jeanette macey simpson,True,0.4
1,aab0162,2010-01-05,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,Amos Ahmed Burch,HardwareEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,jeanette macey simpson,True,0.4
2,aab0162,2010-01-06,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,Amos Ahmed Burch,HardwareEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,jeanette macey simpson,True,0.4
3,aab0162,2010-01-07,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,Amos Ahmed Burch,HardwareEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,jeanette macey simpson,True,0.4
4,aab0162,2010-01-08,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,Amos Ahmed Burch,HardwareEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,jeanette macey simpson,True,0.4


In [8]:
layer_b_path = save_dataset(
    dataset=layer_b_dataset,
    filename=f"ueba_dataset_{MV}b.parquet",
    output_dir=DEFAULT_OUTPUT_DIR
)

Successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6b.parquet


### Creating the Chronological Train/Test Split:

In [9]:
# Splitting dataset B
train_df, test_df = chronological_split(df=layer_b_dataset, split_ratio=0.9)

In [10]:
# Saving training and testing sets
train_path = save_dataset(train_df, f"ueba_dataset_{MV}_train.parquet", DEFAULT_OUTPUT_DIR)
test_path = save_dataset(test_df, f"ueba_dataset_{MV}_test_stream.parquet", DEFAULT_OUTPUT_DIR)

Successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6_train.parquet
Successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6_test_stream.parquet


### Creating the Calibration Split:

In [11]:
train_df, calib_df = chronological_split(df=train_df, split_ratio=8/9)

In [12]:
# Loading insiders
insiders_df = get_insiders(path=INSIDERS_PATH, version=6.2)
insiders_id = set(insiders_df["user"].unique())

In [13]:
# Insider-free set (used for fitting calibration baselines and thresholds)
calib_clean_df = calib_df[~calib_df["user"].isin(insiders_id)].copy()

# Saving both calibration variants
calib_clean_df.to_parquet(UEBA_CALIBRATION_PATH, index=False)
calib_df.to_parquet(UEBA_CALIB_EVAL_PATH, index=False)

# Overwriting train set with the calibration-excluded version
train_df.to_parquet(train_path, index=False)

In [14]:
print(f"Training Set: {len(train_df):,} rows  "
      f"({train_df['day'].min().date()} - {train_df['day'].max().date()})  "
      f"[calibration-excluded]")
print(f"Calibration Clean: {len(calib_clean_df):,} rows  "
      f"({calib_clean_df['day'].min().date()} - {calib_clean_df['day'].max().date()})  "
      f"[insider-free, for baseline fitting]")
print(f"Calibration Eval: {len(calib_df):,} rows  "
      f"({calib_df['day'].min().date()} - {calib_df['day'].max().date()})  "
      f"[with insiders, for AE/IF evaluation]")

Training Set: 1,131,621 rows  (2010-01-02 - 2011-02-19)  [calibration-excluded]
Calibration Clean: 135,356 rows  (2011-02-20 - 2011-04-11)  [insider-free, for baseline fitting]
Calibration Eval: 135,458 rows  (2011-02-20 - 2011-04-11)  [with insiders, for AE/IF evaluation]


###  Utilizing Drill-Down Dataset Lookup:

In [15]:
# Example usage
user = "abh0349"
day = "2010-01-27"

In [16]:
drill_down_table = get_drilldown(layer_a_dataset, user, day)

In [17]:
drill_down_table

,user,pc,day,logon_count,logoff_count,off_hours_logon,logon_late_night_count,logon_hourly_entropy,logon_peak_hour_count,logon_longest_active_run_minutes,...,unique_domains_visited,http_hourly_entropy,http_peak_hour_count,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_used_prior,n_pcs_used_today,new_pc_after_stable_history
11356,abh0349,pc-1019,2010-01-27,2,1,0,0,1,1,0,...,7,1,3,17,1,0.809524,1,5,2,0
11357,abh0349,pc-1534,2010-01-27,1,1,1,0,0,2,1,...,0,0,0,0,0,0.000000,0,5,2,1
